# Towards a GIS system

In [ ]:
#| default_exp database

In [ ]:
#| export
import numpy as np
import sys
import os
import math
from math import radians, cos, sin, sqrt, atan2
import random

#data
from collections import namedtuple
from dataclasses import dataclass,  field, asdict
import json
from typing import List , Optional
from enum import Enum



In [ ]:
#| export
from dataclasses import dataclass
from typing import Optional

from datetime import datetime


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx


In [ ]:
#| export
from importlib import resources
from pathlib import Path
import tempfile
import shutil
import sqlite3
from dataclasses import dataclass
from typing import Iterator
import numpy as np

In [ ]:
#| export
from HexMagic.climate import TerrainPatterns, DrainageBasins, Geology, TerraDemo, Terrain, GeoBounds, ClimatePreset, TerrainFactory
from HexMagic.primitives import MapCord, MapSize, MapRect, MapPath, Hex, HexGrid, HexWrapper, HexPosition, hexBackground, HexRegion, unique_windy_edge, HexChunk
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.cover import ChunkCover, TerraDemo

In [ ]:
#| export
from HexMagic.geology import River, Watershed

In [ ]:
#| export
import asyncio
from dataclasses import dataclass

In [ ]:
#TerraDemo().upscaleWater(TerraDemo().maui_map())

In [ ]:
!cat ../llms.txt

## Helpers

In [ ]:
#| export
LoadResult = namedtuple('LoadResult', ['data', 'status', 'context'])
SaveResult = namedtuple('SaveResult', ['id', 'status', 'context'])

In [ ]:
#| export
@dataclass
class ChunkRef:
    """Reference to a chunk in the world - uses HexPosition for coordinates."""
    position: HexPosition  # Chunk's position in chunk-space (q+r+s=0)
    rings: int             # Hex rings per chunk
    
    @property
    def center_hex(self) -> HexPosition:
        """World hex coordinates of chunk center."""
        spacing = self.rings * 2
        return spacing * self.position
    
    @property
    def key(self) -> tuple[int, int, int]:
        """Hashable key for dict storage."""
        return (self.position.q, self.position.r, self.position.s)
    
    def __hash__(self):
        return hash(self.key)
    
    @classmethod
    def from_key(cls, key, chunk_rings: int) -> 'ChunkRef':
        if isinstance(key, tuple):
            q, r, s = key
        else:
            q, r, s = map(int, key.split(','))
        return cls(HexPosition(q, r, s), chunk_rings)




def chunk_to_world(chunk: ChunkRef, local_pos: HexPosition) -> HexPosition:
    """Map local hex position within chunk to world coordinates."""
    return chunk.center_hex + local_pos


def world_to_chunk(world_pos: HexPosition, chunk_rings: int) -> tuple[HexPosition, HexPosition]:
    """Map world position to (chunk_position, local_position_within_chunk)."""
    spacing = chunk_rings * 2
    
    chunk_q = world_pos.q // spacing
    chunk_r = world_pos.r // spacing
    chunk_s = -chunk_q - chunk_r
    
    chunk_pos = HexPosition(chunk_q, chunk_r, chunk_s)
    chunk_center = spacing * chunk_pos
    local_pos = world_pos - chunk_center
    
    return chunk_pos, local_pos


## Tables

In [ ]:
#| export
@dataclass  
class HexData:
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    grid_index: int = 0
    elevation: float = 0.0
    plate_id:int = -1
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    distance_from_coast: Optional[float] = None
    watershed_id: Optional[int] = None  # NEW: FK to WatershedMeta
    chunk_q: Optional[int] = None  # Chunk origin position
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None
    scale_level: int = 0  # 0=coarse, >0=zoomed scale factor
    modified: int = 0
    last_accessed: int = 0   # Unix timestamp
    access_count: int = 0    # Hit counter


In [ ]:
#| export
@dataclass
class HexWeather:
    """Weather data for a hex at a point in time."""
    id: int = None
    world_id: int = 0
    q: int = 0
    r: int = 0
    s: int = 0
    # Core weather fields
    temperature: float = 0.0           # °C
    precipitation: float = 0.0         # mm/year
    humidity: Optional[float] = None   # %
    # Derived/computed fields
    climate_pet: Optional[float] = None          # potential evapotranspiration
    aridity_index: Optional[float] = None
    temp_seasonality: Optional[float] = None
    precip_seasonality: Optional[float] = None
    distance_from_coast: Optional[float] = None  # hex units
    # Temporal
    season: str = ""                   # e.g., "annual", "summer", "winter"

     # NEW: Chunk identification (NULL for coarse-level weather)
    chunk_q: Optional[int] = None
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None
    
    # NEW: For cache invalidation
    scale_level: int = 0        # 0=coarse, 1=fine chunk
    climate_name: str = ""      # e.g., "mediterranean" - invalidate if changed
    wind_dir: Optional[float] = None  # Wind direction used in computation


    modified: int = 0


In [ ]:
@dataclass
class CacheStats:
    cover_id: int
    chunk_count: int
    total_hexes: int
    oldest_access: int
    newest_access: int
    total_size_mb: float




#| export
@dataclass
class WatershedMeta:
    """Metadata for a stored watershed."""
    id: int = None
    world_id: int = 0
    name: str = ""
    # Terminal hex (outlet)
    terminal_q: int = 0
    terminal_r: int = 0
    terminal_s: int = 0
    is_ocean: bool = True
    # Stats
    total_flow: float = 0.0
    area_hexes: int = 0
    # Encoded data
    river_tree: str = ""   # River.encode()
    style: str = ""        # StyleCSS.encode()
    created: int = 0

    # NEW: Chunk identification (NULL for coarse-level watershed)
    chunk_q: Optional[int] = None
    chunk_r: Optional[int] = None
    chunk_s: Optional[int] = None

    modified: int = 0

In [ ]:
#| export


@dataclass
class World:
    """Metadata for a stored ChunkCober."""
    id: int = None
    name: str = ""
    extras: str = ""
    created: int = 0
    cover_data: str = ""  # Terrain.encode() for coarse map
    modified: int = 0




In [ ]:
#| export
@dataclass
class User:
    username: str
    email: str
    password: str
    created: int  # Unix timestamp
    sessionID: str
    activeWorld: int # a link into world_id for world 
    id: int = None

In [ ]:
#| export
@dataclass
class ChunkBorder:
    """Track drainage across chunk boundaries."""
    id: int = None
    world_id: int = 0
    chunk_q: int = 0
    chunk_r: int = 0  
    chunk_s: int = 0
    border_hex_q: int = 0  # Local hex at border
    border_hex_r: int = 0
    border_hex_s: int = 0
    downstream_chunk_q: int = 0  # Which chunk it drains to
    downstream_chunk_r: int = 0
    downstream_chunk_s: int = 0
    flow_volume: float = 0.0  # Accumulated upstream area


## Main Storage

In [ ]:
#| export
class GeoStorage:

    def __init__(self, custom_path=None):
        path = GeoStorage.get_db_path(custom_path)
        self.path = path
        self.createDB()

    def createDB(self):
        db = database(self.path)
        self.db = db

        # Create tables using dataclasses
        db.create(World, pk='id', if_not_exists=True, transform=True)
        db.create(HexData, pk='id', if_not_exists=True, transform=True)
        db.create(User, pk='id', if_not_exists=True, transform=True)

        db.create(HexWeather, pk='id', if_not_exists=True, transform=True)

        # NEW: Watershed table
        #db.create(WatershedMeta, pk='id', if_not_exists=True, transform=True)
        db.create(ChunkBorder, pk='id', if_not_exists=True, transform=True)

        # After creating tables
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_world ON hex_data(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_coords ON hex_data(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_grid ON hex_data(world_id, grid_index)")


        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_temporal ON hex_data(world_id, q, r, s, modified DESC)")

                # Find borders by source chunk
        db.execute("CREATE INDEX IF NOT EXISTS idx_border_chunk ON chunk_border(world_id, chunk_q, chunk_r, chunk_s)")

        # Find borders draining INTO a chunk (for propagating flow downstream)
        db.execute("CREATE INDEX IF NOT EXISTS idx_border_downstream ON chunk_border(world_id, downstream_chunk_q, downstream_chunk_r, downstream_chunk_s)")


        

         # NEW: Index for watershed queries
        #db.execute("CREATE INDEX IF NOT EXISTS idx_hex_watershed ON hex_data(world_id, watershed_id)")
        #db.execute("CREATE INDEX IF NOT EXISTS idx_watershed_world ON watershed_meta(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_hex_chunk ON hex_data(world_id, chunk_q, chunk_r, chunk_s, scale_level)")

        # Indices for weather queries
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_world ON hex_weather(world_id)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_coords ON hex_weather(world_id, q, r, s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_temporal ON hex_weather(world_id, q, r, s, modified DESC)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_season ON hex_weather(world_id, season)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_chunk ON hex_weather(world_id, chunk_q, chunk_r, chunk_s)")
        db.execute("CREATE INDEX IF NOT EXISTS idx_weather_scale ON hex_weather(world_id, scale_level)")


        self.weather = db.t.hex_weather

        

        # Store table references using db.t notation
        self.users = db.t.user
        self.hexes = db.t.hex_data
        self.worlds = db.t.world
        #self.watersheds = db.t.watershed_meta  # Add table reference
        self.borders = db.t.chunk_border  # Add table reference

        
        
        # Populate templates (after tables exist)
        #self.populate_templates()


    @staticmethod
    def get_db_path(custom_path=None):
        """Get database path. Tries package data, falls back to user directory."""
        if custom_path:
            return custom_path
        
        # Try package data directory first
        try:
            db_dir = resources.files('HexMagic').joinpath('data/db')
            db_path = Path(db_dir) / 'hexmagic.db'
            # Test if writable
            db_path.parent.mkdir(parents=True, exist_ok=True)
            db_path.touch(exist_ok=True)
            return str(db_path)
        except (PermissionError, OSError):
            # Fallback to user directory
            data_dir = Path.home() / '.hexmagic' / 'data'
            data_dir.mkdir(parents=True, exist_ok=True)
            return str(data_dir / 'hexmagic.db')

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.db.close()

    
    @staticmethod
    def _get(obj, key):
        """Get attribute from dict or object."""
        return obj[key] if isinstance(obj, dict) else getattr(obj, key)

In [ ]:
#| export


#this might be something we don't export
@patch
def reset_database(self:GeoStorage):
    """Delete and recreate database. USE WITH CAUTION!"""
    db_path = Path(self.path)  # Convert string to Path
    
    if db_path.exists():
        db_path.unlink()

    self.createDB()



In [ ]:
#| export

@patch
def _latest_hex_subquery(self: GeoStorage, world_id: int, as_of: int = None) -> str:
    """SQL subquery for latest hex state, optionally at a point in time."""
    time_clause = f"AND modified <= {as_of}" if as_of else ""
    return f"""
        SELECT world_id, q, r, s, MAX(modified) as max_mod
        FROM hex_data
        WHERE world_id = {world_id} {time_clause}
        GROUP BY world_id, q, r, s
    """




In [ ]:
#| export
@patch
def query_hexes_latest(self: GeoStorage, world_id: int, as_of: int = None) -> list:
    """Get most recent version of each hex, optionally at a specific time."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    cursor = self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
    """)
    
    # Get column names from cursor description
    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    
    return rows


In [ ]:
#| export
@patch
def update_hex(self: GeoStorage, world_id: int, q: int, r: int, s: int,
               elevation: float = None, latitude: float = None, longitude: float = None,
               distance_from_coast: float = None, grid_index: int = None) -> SaveResult:
    """Append a new state for a hex (temporal update)."""
    now = int(datetime.now().timestamp())
    
    record = {
        'world_id': world_id,
        'q': q, 'r': r, 's': s,
        'modified': now
    }
    
    # Get previous state to preserve unchanged fields
    prev = self.query_hexes_in_radius(world_id, q, r, s, radius=0)
    if prev:
        prev = prev[0]
        record['grid_index'] = grid_index if grid_index is not None else GeoStorage._get(prev, 'grid_index')
        record['elevation'] = elevation if elevation is not None else GeoStorage._get(prev, 'elevation')
        record['latitude'] = latitude if latitude is not None else GeoStorage._get(prev, 'latitude')
        record['longitude'] = longitude if longitude is not None else GeoStorage._get(prev, 'longitude')
        record['distance_from_coast'] = distance_from_coast if distance_from_coast is not None else GeoStorage._get(prev, 'distance_from_coast')
    else:
        record['grid_index'] = grid_index or 0
        record['elevation'] = elevation or 0.0
        record['latitude'] = latitude
        record['longitude'] = longitude
        record['distance_from_coast'] = distance_from_coast
    
    try:
        self.hexes.insert(record)
        return SaveResult(None, 'updated', f'hex ({q},{r},{s}) at {now}')
    except Exception as e:
        return SaveResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    rows = list(self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ]).fetchall())
    
    if not rows:
        return []
    
    # Hardcode column order to avoid cursor.description issue
    cols = ['id', 'world_id', 'q', 'r', 's', 'grid_index', 'elevation', 
        'plate_id', 'latitude', 'longitude', 'distance_from_coast', 
        'watershed_id', 'modified']
    
    rows = [dict(zip(cols, row)) for row in rows]
    
    # Filter to actual hex distance
    return [row for row in rows
            if max(abs(row['q'] - center_q), abs(row['r'] - center_r), abs(row['s'] - center_s)) <= radius]


### World

In [ ]:
#| export
@patch
def save_cover(self: GeoStorage, cover: ChunkCover, name: str = "") -> SaveResult:
    """Save ChunkCover to database.
    
    Args:
        cover: ChunkCover to save
        name: Optional name for the world
    
    Returns:
        SaveResult(world_id, status, context)
    """
    now = int(datetime.now().timestamp())
    
    try:
        with self.db.conn:
            # Handle ID assignment
            if cover.ident < 0:
                # Get next available ID
                cursor = self.db.execute("SELECT MAX(id) FROM world")
                max_id = cursor.fetchone()[0]
                cover.ident = (max_id or 0) + 1
            
            # Check if exists
            existing = self.db.execute(
                "SELECT id FROM world WHERE id = ?", 
                [cover.ident]
            ).fetchone()
            
            encoded = cover.encode()
            
            record = {
                'id': cover.ident,
                'name': name or f"Cover_{cover.ident}",
                'cover_data': encoded,
                'modified': now
            }
            
            if existing:
                # Update existing
                record['created'] = self.worlds[cover.ident]['created']
                self.worlds.update(record)
                status_msg = 'updated'
            else:
                # Insert new
                record['created'] = now
                self.worlds.insert(record)
                status_msg = 'saved'
            
            return SaveResult(
                cover.ident, 
                status_msg, 
                f"ChunkCover {cover.rings}r+{cover.halo_rings}h, {len(cover.terrain.hexGrid.hexes)} hexes"
            )
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_cover(self: GeoStorage, cover_id: int) -> LoadResult:
    """Load ChunkCover from database.
    
    Args:
        cover_id: ID of the cover to load
    
    Returns:
        LoadResult(ChunkCover, status, context)
    """
    try:
        world = self.worlds[cover_id]
        if not world:
            return LoadResult(None, 'not_found', f'Cover {cover_id} not found')
        
        cover_data = GeoStorage._get(world, 'cover_data')
        if not cover_data:
            return LoadResult(None, 'error', 'No cover_data in world record')
        
        cover = ChunkCover.decode(cover_data)
        cover.ident = cover_id
        cover.db = self  # Attach database reference
        
        name = GeoStorage._get(world, 'name')
        return LoadResult(
            cover, 
            'loaded', 
            f"{name}: {cover.rings}r+{cover.halo_rings}h, {len(cover.terrain.hexGrid.hexes)} hexes"
        )
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def save(self: ChunkCover, name: str = "") -> SaveResult:
    """Save this ChunkCover to its attached database.
    
    Args:
        name: Optional name for the world
    
    Returns:
        SaveResult with updated ident
    """
    if self.db is None:
        return SaveResult(None, 'error', 'No database attached to ChunkCover')
    
    result = self.db.save_cover(self, name=name)
    
    # Update ident if it was assigned
    if result.status in ('saved', 'updated') and result.id is not None:
        self.ident = result.id
    
    return result


## Chunk

In [ ]:
#| export
@patch
def load_or_generate_chunk(self: GeoStorage, 
                           cover: ChunkCover,
                           origin: int = None,
                           scale: int = 2,
                           method: str = 'bilinear',
                           octaves: int = 2,
                           persistence: float = 0.3,
                           detail_strength: float = 0.2,
                           force_regenerate: bool = False) -> LoadResult:
    """Load cached chunk or generate new one.
    
    Returns LoadResult(terrain, status, context) where:
    - status='cached' if loaded from DB
    - status='generated' if computed fresh
    """
    # 1. Compute chunk identifier
    grid = cover.terrain.hexGrid
    if origin is None:
        origin = grid.middle
    
    origin_pos = grid.index_to_hexposition(origin)
    chunk_key = f"{cover.ident}_{origin_pos.q}_{origin_pos.r}_{origin_pos.s}_{scale}"
    
    # 2. Check cache
    if not force_regenerate:
        cached = self._load_cached_chunk(cover.ident, origin_pos, scale)
        if cached.status == 'loaded':
            return LoadResult(cached.data, 'cached', cached.context)
    
    # 3. Generate chunk
    zoomed = cover.zoomChunkCombined(
        origin=origin,
        scale=scale,
        method=method,
        octaves=octaves,
        persistence=persistence,
        detail_strength=detail_strength
    )
    
    # 4. Save to cache
    save_result = self._save_chunk_cache(
        cover.ident, 
        origin_pos, 
        scale, 
        zoomed
    )
    # Periodic eviction check (every N generations)
    if random.random() < 0.1:  # 10% of the time
        self.evict_lru_chunks(cover.ident, max_chunks=100)
    
    return LoadResult(
        zoomed, 
        'generated', 
        f"{save_result.context}, scale={scale}"
    )





@patch
def _save_chunk_cache(self: GeoStorage,
                      cover_id: int,
                      origin_pos: HexPosition,
                      scale: int,
                      terrain: Terrain) -> SaveResult:
    """Save zoomed terrain as cached chunk."""
    now = int(datetime.now().timestamp())
    grid = terrain.hexGrid
    
    try:
        with self.db.conn:
            count = 0
            for idx in range(len(terrain.elevations)):
                # Skip invalid hexes
                if idx in grid.invalidRegion:
                    continue
                
                pos = grid.index_to_hexposition(idx)
                
                record = {
                    'world_id': cover_id,
                    'q': pos.q,
                    'r': pos.r,
                    's': pos.s,
                    'chunk_q': origin_pos.q,
                    'chunk_r': origin_pos.r,
                    'chunk_s': origin_pos.s,
                    'grid_index': idx,
                    'elevation': float(terrain.elevations[idx]),
                    'scale_level': scale,
                    'modified': now
                }
                self.hexes.insert(record)
                count += 1
            
            return SaveResult(count, 'saved', f'{count} hexes cached')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def invalidate_chunk_cache(self: GeoStorage, cover_id: int) -> SaveResult:
    """Delete all cached chunks for a cover (when coarse terrain changes)."""
    try:
        cursor = self.db.execute("""
            DELETE FROM hex_data 
            WHERE world_id = ? AND scale_level > 0
        """, [cover_id])
        
        count = cursor.rowcount
        return SaveResult(count, 'invalidated', f'{count} cached hexes deleted')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def _load_cached_chunk(self: GeoStorage, 
                       cover_id: int,
                       origin_pos: HexPosition,
                       scale: int) -> LoadResult:
    """Load cached chunk hexes from database."""
    try:
        # Query hexes with matching chunk coordinates
        cursor = self.db.execute("""
            SELECT h.* FROM hex_data h
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_data
                WHERE world_id = ? 
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level = ?
                GROUP BY world_id, q, r, s
            ) latest 
                ON h.world_id = latest.world_id 
                AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
                AND h.modified = latest.max_mod
        """, [cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        cols = [d[0] for d in cursor.description]
        rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
        
        if not rows:
            return LoadResult(None, 'not_found', 'No cached chunk')

        if rows:  # Cache hit - update access stats
            now = int(datetime.now().timestamp())
            self.db.execute("""
                UPDATE hex_data 
                SET last_accessed = ?, access_count = access_count + 1
                WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                AND scale_level = ?
            """, [now, cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        # Reconstruct terrain from rows
        # (You'll need to infer grid dimensions from the data)
        terrain = self._reconstruct_terrain_from_rows(rows, cover_id)
        
        return LoadResult(terrain, 'loaded', f'{len(rows)} cached hexes')
    
    except Exception as e:
        return LoadResult(None, 'error', str(e))

In [ ]:
#| export
@patch
def evict_lru_chunks(self: GeoStorage, cover_id: int, 
                     max_chunks: int = 100) -> SaveResult:
    """Evict oldest chunks until under limit."""
    
    # Count current chunks
    count = self.db.execute("""
        SELECT COUNT(DISTINCT chunk_q || ',' || chunk_r || ',' || chunk_s) 
        FROM hex_data WHERE world_id = ? AND scale_level > 0
    """, [cover_id]).fetchone()[0]
    
    if count <= max_chunks:
        return SaveResult(0, 'ok', 'under limit')
    
    to_evict = count - max_chunks
    
    # Find oldest chunks
    oldest = self.db.execute("""
        SELECT DISTINCT chunk_q, chunk_r, chunk_s, MIN(last_accessed) as oldest
        FROM hex_data 
        WHERE world_id = ? AND scale_level > 0
        GROUP BY chunk_q, chunk_r, chunk_s
        ORDER BY oldest ASC
        LIMIT ?
    """, [cover_id, to_evict]).fetchall()
    
    # Delete them
    deleted = 0
    with self.db.conn:
        for row in oldest:
            self.db.execute("""
                DELETE FROM hex_data 
                WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level > 0
            """, [cover_id, row['chunk_q'], row['chunk_r'], row['chunk_s']])
            deleted += 1
    
    return SaveResult(deleted, 'evicted', f'{deleted} chunks removed')


### Region

In [ ]:
#| export
@patch
def query_hexes_in_radius(self: GeoStorage, world_id: int,
                          center_q: int, center_r: int, center_s: int,
                          radius: int, as_of: int = None) -> list:
    """Query latest hexes within radius of a center position."""
    subq = self._latest_hex_subquery(world_id, as_of)
    
    cursor = self.db.execute(f"""
        SELECT h.* FROM hex_data h
        INNER JOIN ({subq}) latest 
            ON h.world_id = latest.world_id 
            AND h.q = latest.q AND h.r = latest.r AND h.s = latest.s
            AND h.modified = latest.max_mod
        WHERE h.q BETWEEN ? AND ? AND h.r BETWEEN ? AND ?
    """, [
        center_q - radius, center_q + radius,
        center_r - radius, center_r + radius
    ])
    
    # Convert to dicts like query_hexes_latest does
    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    
    # Filter to actual hex distance
    result = []
    for row in rows:
        dq = abs(GeoStorage._get(row, 'q') - center_q)
        dr = abs(GeoStorage._get(row, 'r') - center_r)
        ds = abs(GeoStorage._get(row, 's') - center_s)
        dist = max(dq, dr, ds)
        if dist <= radius:
            result.append(row)
    
    return result


### Weather

In [ ]:
#| export
@patch
def save_zoomed_weather(self: GeoStorage, 
                        cover_id: int,
                        origin_pos: HexPosition,
                        scale: int,
                        terrain: Terrain,
                        climate_name: str = "",
                        season: str = "annual") -> SaveResult:
    """Save weather from zoomed terrain, tagged by chunk origin."""
    now = int(datetime.now().timestamp())
    grid = terrain.hexGrid
    invalid = getattr(grid, 'invalidRegion', set())
    
    if 'temperature' not in terrain.fields or 'precipitation' not in terrain.fields:
        return SaveResult(None, 'error', 'Missing weather fields')
    
    try:
        with self.db.conn:
            count = 0
            for idx in range(len(terrain.elevations)):
                if idx in invalid:
                    continue
                
                pos = grid.index_to_hexposition(idx)
                
                record = {
                    'world_id': cover_id,
                    'q': pos.q,
                    'r': pos.r,
                    's': pos.s,
                    'chunk_q': origin_pos.q,
                    'chunk_r': origin_pos.r,
                    'chunk_s': origin_pos.s,
                    'temperature': float(terrain.fields['temperature'][idx]),
                    'precipitation': float(terrain.fields['precipitation'][idx]),
                    'climate_name': climate_name,
                    'scale_level': scale,
                    'season': season,
                    'modified': now
                }
                
                # Add optional fields
                for field in ['humidity', 'climate_pet', 'aridity_index']:
                    if field in terrain.fields:
                        record[field] = float(terrain.fields[field][idx])
                
                self.weather.insert(record)
                count += 1
            
            return SaveResult(count, 'saved', f'{count} weather records for chunk ({origin_pos.q},{origin_pos.r},{origin_pos.s})')
    
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_zoomed_weather(self: GeoStorage,
                        cover_id: int,
                        origin_pos: HexPosition,
                        scale: int,
                        terrain: Terrain,
                        season: str = "annual") -> LoadResult:
    """Load cached weather into zoomed terrain."""
    try:
        cursor = self.db.execute("""
            SELECT hw.* FROM hex_weather hw
            INNER JOIN (
                SELECT world_id, q, r, s, MAX(modified) as max_mod
                FROM hex_weather
                WHERE world_id = ? AND season = ?
                  AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
                  AND scale_level = ?
                GROUP BY world_id, q, r, s
            ) latest 
                ON hw.world_id = latest.world_id 
                AND hw.q = latest.q AND hw.r = latest.r AND hw.s = latest.s
                AND hw.modified = latest.max_mod
        """, [cover_id, season, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        cols = [d[0] for d in cursor.description]
        rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
        
        if not rows:
            return LoadResult(terrain, 'not_found', 'No cached weather')
        
        # Initialize fields
        n = len(terrain.elevations)
        for field in ['temperature', 'precipitation', 'humidity', 'climate_pet', 'aridity_index']:
            if field not in terrain.fields:
                terrain.fields[field] = np.zeros(n)
        
        grid = terrain.hexGrid
        filled = 0
        
        for row in rows:
            pos = HexPosition(row['q'], row['r'], row['s'])
            idx = grid.hexposition_to_index(pos)
            
            if 0 <= idx < n:
                terrain.fields['temperature'][idx] = row['temperature'] or 0.0
                terrain.fields['precipitation'][idx] = row['precipitation'] or 0.0
                if row.get('humidity') is not None:
                    terrain.fields['humidity'][idx] = row['humidity']
                if row.get('climate_pet') is not None:
                    terrain.fields['climate_pet'][idx] = row['climate_pet']
                if row.get('aridity_index') is not None:
                    terrain.fields['aridity_index'][idx] = row['aridity_index']
                filled += 1
        
        return LoadResult(terrain, 'loaded', f'{filled} weather records')
    
    except Exception as e:
        return LoadResult(terrain, 'error', str(e))


@patch
def has_zoomed_weather(self: GeoStorage,
                       cover_id: int,
                       origin_pos: HexPosition,
                       scale: int,
                       season: str = "annual",
                       min_count: int = 1) -> bool:
    """Check if cached weather exists for this zoom."""
    try:
        cursor = self.db.execute("""
            SELECT COUNT(*) FROM hex_weather
            WHERE world_id = ? AND season = ?
              AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
              AND scale_level = ?
        """, [cover_id, season, origin_pos.q, origin_pos.r, origin_pos.s, scale])
        
        count = cursor.fetchone()[0]
        return count >= min_count
    except Exception:
        return False


### watershed

### Chunk Cover

In [ ]:
#| export
@patch
def _reconstruct_terrain_from_rows(self: GeoStorage, rows: list[dict], cover_id: int) -> Terrain:
    """Reconstruct Terrain from cached hex_data rows."""
    if not rows:
        return None
    
    # Infer grid dimensions from row data
    qs = [r['q'] for r in rows]
    rs = [r['r'] for r in rows]
    
    # Get original cover to retrieve grid params
    world = self.worlds[cover_id]
    cover_data = GeoStorage._get(world, 'cover_data')
    if cover_data:
        # Extract just the grid info from cover
        original_cover = ChunkCover.decode(cover_data)
        base_radius = original_cover.terrain.hexGrid.radius
    else:
        base_radius = 10  # fallback
    
    # Compute grid size from coordinate ranges
    nRows = max(rs) - min(rs) + 1
    nCols = max(qs) - min(qs) + 1
    
    # Scale factor stored in rows
    scale = rows[0].get('scale_level', 1)
    radius = base_radius / scale
    
    grid = HexGrid(nRows=nRows, nCols=nCols, radius=radius, style=StyleCSS("simple"))
    
    terrain = Terrain.__new__(Terrain)
    terrain.hexGrid = grid
    terrain.elevations = np.zeros(len(grid.hexes))
    terrain.fields = {}
    
    # Map rows to terrain
    for row in rows:
        pos = HexPosition(row['q'], row['r'], row['s'])
        idx = grid.hexposition_to_index(pos)
        if 0 <= idx < len(terrain.elevations):
            terrain.elevations[idx] = row['elevation']
    
    return terrain


In [ ]:
#| export
@patch
def zoom_cached(self: ChunkCover, 
                origin: int = None,
                scale: int = 2,
                **kwargs) -> Terrain:
    """Zoom with database caching (requires db attached)."""
    if self.db is None:
        # Fallback to direct generation
        return self.zoomChunkCombined(origin=origin, scale=scale, **kwargs)
    
    result = self.db.load_or_generate_chunk(
        self, 
        origin=origin, 
        scale=scale, 
        **kwargs
    )
    
    if result.status == 'cached':
        print(f"✓ Loaded from cache: {result.context}")
    else:
        print(f"⚙ Generated: {result.context}")
    
    return result.data


In [ ]:
#| export
@patch
def zoom_with_full_data(self: ChunkCover,
                        origin: int = None,
                        scale: int = 2,
                        compute_weather: bool = True,
                        force_regenerate: bool = False) -> DrainageBasins:
    """Zoom chunk with weather and watersheds, using cache where possible.
    
    Returns DrainageBasins with terrain set to the zoomed terrain.
    """
    if self.db is None:
        raise ValueError("ChunkCover.db must be set for caching")
    
    grid = self.terrain.hexGrid
    if origin is None:
        origin = grid.middle
    origin_pos = grid.index_to_hexposition(origin)
    
    # Step 1: Lazy init coarse basin (O(n²) only once)
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        # Save cover with basin to persist
        self.save()
    
    # Step 2: Base terrain (cached)
    result = self.db.load_or_generate_chunk(
        self, origin=origin, scale=scale, force_regenerate=force_regenerate
    )
    zoomed = result.data
    
    # Step 3: Weather
    if compute_weather:
        if not force_regenerate and self.db.has_zoomed_weather(self.ident, origin_pos, scale):
            self.db.load_zoomed_weather(self.ident, origin_pos, scale, zoomed)
        else:
            zoomed.climate = self.terrain.climate
            zoomed.geo = self.terrain.geo
            zoomed.compute_weather()
            self.db.save_zoomed_weather(self.ident, origin_pos, scale, zoomed)
    
    # Step 4: Build fine_regions mapping (O(n), cheap)
    fine_regions = self._build_fine_regions_map(origin, scale, zoomed.hexGrid)
    
    # Step 5: Project watersheds (O(n), uses cached coarse basin)
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, zoomed)
    
    # Step 6: Return DrainageBasins with zoomed terrain
    zoomed_basins = DrainageBasins(zoomed, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return zoomed_basins


@patch
def _build_fine_regions_map(self: ChunkCover, 
                            origin: int, 
                            scale: int, 
                            fine_grid: HexGrid) -> dict[int, set[int]]:
    """Map coarse hex indices to sets of fine hex indices.
    
    O(n) — just coordinate math, no flow computation.
    """
    coarse_grid = self.terrain.hexGrid
    extent = 3 * self.rings + self.halo_rings
    
    center_row = origin // coarse_grid.nCols
    center_col = origin % coarse_grid.nCols
    
    fine_regions = {}
    invalid = getattr(fine_grid, 'invalidRegion', set())
    
    for coarse_row in range(center_row - extent, center_row + extent + 1):
        for coarse_col in range(center_col - extent, center_col + extent + 1):
            if not (0 <= coarse_row < coarse_grid.nRows and 0 <= coarse_col < coarse_grid.nCols):
                continue
            
            coarse_idx = coarse_row * coarse_grid.nCols + coarse_col
            
            fine_start_row = (coarse_row - (center_row - extent)) * scale
            fine_start_col = (coarse_col - (center_col - extent)) * scale
            
            fine_hexes = set()
            for dr in range(scale):
                for dc in range(scale):
                    fine_row = fine_start_row + dr
                    fine_col = fine_start_col + dc
                    if 0 <= fine_row < fine_grid.nRows and 0 <= fine_col < fine_grid.nCols:
                        fine_idx = fine_row * fine_grid.nCols + fine_col
                        if fine_idx not in invalid:
                            fine_hexes.add(fine_idx)
            
            if fine_hexes:
                fine_regions[coarse_idx] = fine_hexes
    
    return fine_regions


In [ ]:
#| export
@patch
def invalidate_all_caches(self: GeoStorage, cover_id: int) -> dict:
    """Invalidate all cached data when coarse terrain changes."""
    results = {}

     # Use transaction for atomic invalidation
    try:
        with self.db.conn:
            
            # Invalidate terrain chunks
            results['terrain'] = self.invalidate_chunk_cache(cover_id)
            
            # Invalidate weather
            cursor = self.db.execute("""
                DELETE FROM hex_weather 
                WHERE world_id = ? AND scale_level > 0
            """, [cover_id])
            results['weather'] = SaveResult(cursor.rowcount, 'invalidated', f'{cursor.rowcount} weather records')
            
            # Invalidate watersheds
            cursor = self.db.execute("""
                DELETE FROM hex_data 
                WHERE world_id = ? AND scale_level > 0 AND watershed_id IS NOT NULL
            """, [cover_id])
            results['watersheds'] = SaveResult(cursor.rowcount, 'invalidated', f'{cursor.rowcount} watershed assignments')
        
            return results
    except Exception as e:
        # Transaction rolled back automatically
        return {'error': SaveResult(None, 'error', str(e))}


In [ ]:
#| export
# === WATERSHED PERSISTENCE ===

@patch
def save_chunk_watersheds(self: GeoStorage, chunk: HexChunk, 
                          chunk_ref: ChunkRef, world_id: int) -> SaveResult:
    """Save watershed assignments for a chunk."""
    now = int(datetime.now().timestamp())
    p = chunk_ref.position
    
    if 'watershed_id' not in chunk.fields:
        return SaveResult(None, 'error', 'Chunk missing watershed_id field')
    
    try:
        with self.db.conn:
            count = 0
            for idx in chunk.iter_core():
                world_pos = chunk.index_to_world(idx)
                ws_id = int(chunk.fields['watershed_id'][idx])
                
                if ws_id < 0:
                    continue  # Skip unassigned/ocean
                
                # Update hex_data with watershed_id
                self.hexes.insert({
                    'world_id': world_id,
                    'q': world_pos.q,
                    'r': world_pos.r,
                    's': world_pos.s,
                    'grid_index': idx,
                    'elevation': float(chunk.elevations[idx]),
                    'watershed_id': ws_id,
                    'modified': now
                })
                count += 1
            
            return SaveResult(count, 'saved', 
                            f'{count} watershed assignments for chunk ({p.q},{p.r},{p.s})')
    except Exception as e:
        return SaveResult(None, 'error', str(e))


@patch
def load_chunk_watersheds(self: GeoStorage, world_id: int,
                          chunk: HexChunk, chunk_ref: ChunkRef) -> LoadResult:
    """Load watershed assignments for a chunk."""
    try:
        p = chunk_ref.position
        
        # Query hexes with watershed_id for this chunk's world positions
        hex_rows = []
        for idx in chunk.iter_core():
            world_pos = chunk.index_to_world(idx)
            rows = self.query_hexes_in_radius(world_id, world_pos.q, world_pos.r, world_pos.s, radius=0)
            if rows:
                hex_rows.append((idx, rows[0]))
        
        if not hex_rows:
            return LoadResult(chunk, 'not_found', 
                            f'No watershed data for chunk ({p.q},{p.r},{p.s})')
        
        # Initialize field if needed
        if 'watershed_id' not in chunk.fields:
            chunk.add_field('watershed_id', default=-1)
        
        filled = 0
        for idx, row in hex_rows:
            ws_id = row.get('watershed_id')
            if ws_id is not None and ws_id >= 0:
                chunk.fields['watershed_id'][idx] = ws_id
                filled += 1
        
        return LoadResult(chunk, 'loaded', 
                         f'{filled} watershed assignments for chunk ({p.q},{p.r},{p.s})')
    except Exception as e:
        return LoadResult(None, 'error', str(e))


In [ ]:
#| export
@patch
def has_chunk_watersheds(self: GeoStorage, world_id: int, 
                         chunk_ref: ChunkRef,
                         min_count: int = 1) -> bool:
    """Check if watershed data exists for chunk."""
    try:
        center = chunk_ref.center_hex
        # Use larger radius to cover more of the chunk
        radius = chunk_ref.rings  # Full chunk radius, not just 3
        
        rows = self.query_hexes_in_radius(world_id, center.q, center.r, center.s, radius=radius)
        
        ws_count = sum(1 for r in rows if r.get('watershed_id') is not None and r.get('watershed_id') >= 0)
        return ws_count >= min_count
    except Exception:
        return False



## Debugger

In [ ]:
#| export
class GeoStorageDebugger:
    """Test harness for HexServer with automatic cleanup."""
    
    def __init__(self, keep_on_error=False):
        """
        Args:
            keep_on_error: If True, preserve database when tests fail
        """
        self.temp_dir = tempfile.mkdtemp(prefix='hexmagic_test_')
        self.db_path = os.path.join(self.temp_dir, 'test.db')
        self.server = GeoStorage(custom_path=self.db_path)
        self.keep_on_error = keep_on_error
        self.failed = False
        
    def preserve_db(self, test_name):
        """Copy database to inspection directory."""
        inspect_dir = Path.home() / '.hexmagic' / 'test_failures'
        inspect_dir.mkdir(parents=True, exist_ok=True)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        dest = inspect_dir / f"{test_name}_{timestamp}.db"
        
        shutil.copy2(self.db_path, dest)
        print(f"⚠ Database preserved at: {dest}")
        
    def close(self):
        """Clean up test database."""
        self.server.db.close()
        
        if not self.failed or not self.keep_on_error:
            shutil.rmtree(self.temp_dir)
            print("✓ Test database cleaned up")
        else:
            print(f"⚠ Test database kept at: {self.temp_dir}")
            
    def __enter__(self):
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.failed = True
        self.close()


## The Big refactor

In [ ]:
# Test save/load with GeoStorageDebugger
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Create terrain and cover
    terrain = TerraDemo().maui_map()
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server  # Attach database
    
    print(f"Original: {len(terrain.hexGrid.hexes)} hexes, {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols}")
    
    # 2. Save
    result = cover.save(name="Maui Test")
    print(f"Save: {result.status} - {result.context}")
    cover_id = result.id
    
    # 3. Load back
    loaded = dbg.server.load_cover(cover_id)
    print(f"Load: {loaded.status} - {loaded.context}")
    
    # 4. Verify
    orig_elev = cover.terrain.elevations
    load_elev = loaded.data.terrain.elevations
    
    match = np.allclose(orig_elev, load_elev)
    print(f"Elevations match: {match}")
    
    if match:
        print("✓ Round-trip test passed!")
    else:
        dbg.failed = True
        print("✗ Data mismatch!")


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Maui Chunk Test")
    
    # First zoom - should generate
    result1 = dbg.server.load_or_generate_chunk(cover, origin=None, scale=2)
    print(f"First call: {result1.status} - {result1.context}")
    
    # Second zoom - should hit cache
    result2 = dbg.server.load_or_generate_chunk(cover, origin=None, scale=2)
    print(f"Second call: {result2.status} - {result2.context}")
    
    if result1.status == 'generated' and result2.status == 'cached':
        print("✓ Chunk caching works!")
    else:
        print("✗ Caching issue")


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    terrain.compute_weather()  # Need weather for full data
    
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Maui Full Test")
    
    print(f"Basin before: {cover.basin}")
    
    # zoom_with_full_data should:
    # 1. Lazy-init basin (O(n²) once)
    # 2. Generate zoomed terrain
    # 3. Project watersheds (O(n))
    basins = cover.zoom_with_full_data(origin=None, scale=2, compute_weather=True)
    
    print(f"Basin after: {cover.basin is not None}")
    print(f"Zoomed terrain: {basins.terrain.hexGrid.nRows}x{basins.terrain.hexGrid.nCols}")
    print(f"Watersheds: {len(basins.sheds)}")
    
    # Check weather was computed
    has_temp = 'temperature' in basins.terrain.fields
    print(f"Has temperature: {has_temp}")
    
    if cover.basin is not None and len(basins.sheds) > 0 and has_temp:
        print("✓ zoom_with_full_data works!")
    else:
        print("✗ Something missing")


In [ ]:
with GeoStorageDebugger(keep_on_error=True) as dbg:
    terrain = TerraDemo().maui_map()
    terrain.compute_weather()  # Need weather for full data
    
    cover = ChunkCover(terrain, rings=3, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Maui Full Test")
    
    print(f"Basin before: {cover.basin}")
    
    # zoom_with_full_data should:
    # 1. Lazy-init basin (O(n²) once)
    # 2. Generate zoomed terrain
    # 3. Project watersheds (O(n))
    basins = cover.zoom_with_full_data(origin=None, scale=2, compute_weather=True)
    
    print(f"Basin after: {cover.basin is not None}")
    print(f"Zoomed terrain: {basins.terrain.hexGrid.nRows}x{basins.terrain.hexGrid.nCols}")
    print(f"Watersheds: {len(basins.sheds)}")
    
    # Check weather was computed
    has_temp = 'temperature' in basins.terrain.fields
    print(f"Has temperature: {has_temp}")
terrain = basins.terrain
terrain.colorMap()
terrain.hexGrid.update()
#terrain.hexGrid.builder.show()

    
   


## By Region

In [ ]:
!cat ../HexMagic/plot/*.py

In [ ]:
??HexRegion.crop_to_centered_grid

I want to build something similar to crop_to_centered_grid but using the mainTable and the database. so instead of creating a subsection of the main we would return a detail terrain, drainagebasin,and mapping that would link the new terrain back to hexes on the original terrain. Does this make sense?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, that makes sense! You want something like:

```python
def zoom_region(region: HexRegion, storage: GeoStorage, cover: ChunkCover, scale: int = 2) 
    -> tuple[Terrain, DrainageBasins, callable]:
```

Where the callable would be an index mapper: `fine_idx -> coarse_idx`

A few clarifying questions:

1. **Input**: Would you start from a `HexRegion` on the coarse terrain, or from a center hex + radius?

2. **Caching**: Should this use your existing chunk cache (`load_or_generate_chunk`), or is this for arbitrary regions that might span multiple chunks?

3. **Return mapping**: Do you need bidirectional mapping (fine→coarse AND coarse→fine), or just one direction?

Lets input based upon a HexRegion. we would need to figure out which chunks. I think just a fine->coarse since the reverse will have many destinations


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call on fine→coarse only. Let's think through the steps:

1. **Region → Chunks**: Given a `HexRegion`, figure out which `ChunkRef`s cover it
2. **Load/generate** each chunk (using your existing cache)
3. **Stitch** if multiple chunks (or just return single chunk terrain)
4. **Build mapper**: `fine_idx → coarse_idx`

A question first: How do you want to handle regions that span multiple chunks?

- **Option A**: Always return a single merged terrain (more complex, but seamless)
- **Option B**: Return the smallest single chunk that contains the region's centroid (simpler, might clip edges)
- **Option C**: Return a list of `(chunk_terrain, chunk_mapper)` tuples (keeps chunks separate)

Which approach fits your use case better?

I think we want option A. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Option A it is. Let's break this into pieces.

**Step 1**: Figure out which `ChunkRef`s a region touches.

The key insight: each coarse hex belongs to exactly one chunk. So we can iterate over `region.hexes`, convert each to a chunk position, and collect the unique chunks.

Here's the signature I'm thinking:

```python
def region_to_chunks(region: HexRegion, cover: ChunkCover) -> set[ChunkRef]:
```

Does that look right? Do you want to try implementing it using your existing `world_to_chunk` helper, or should I sketch it out?

So what I am thinking is pick the first hex in the region and find its region. then uses set difference operator to remove all of the hexes in the region that are in that chunk. continue until we have the chunks. then the question is how do we reassemble them. maybe we can take advanage of the cubecoordinates to figure out the average of the chunks which would be our orign. we can figure out the max east and west using the relavivet postions. north and south are harder and we might have to do some y coordinate math to get those.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good algorithm for collecting chunks! The set-difference approach is clean.

For stitching, your intuition about cube coordinates is right. Since `ChunkRef.position` is already a `HexPosition`, you can:

- **Center**: `HexPosition.average(chunk_positions)` 
- **Extent in q/r/s**: `max_q - min_q`, etc.

Here's the thing about north/south: in cube coordinates, **r** roughly corresponds to vertical (north/south). So `min(r)` = northmost chunk, `max(r)` = southmost.

The trickier part is creating the stitched grid. Two options:

**A)** Create a new `HexGrid` sized to fit all chunks, then copy data in using coordinate math

**B)** Create a new `Terrain` directly by concatenating/merging the chunk terrains' elevation arrays

Which feels more natural given your existing code? Or would you like to sketch out option A's grid sizing first?

I think b is better. The goal is to get to a terrain and have a drainage basin (using the chunkcover approach so we don't do o n^2 operations

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Got it. So the flow is:

1. **Collect chunks** covering the region
2. **Load/generate** each chunk terrain (cached)
3. **Stitch** into one merged `Terrain`
4. **Project watersheds** using existing `cover.project_watersheds()` (O(n), not O(n²))
5. **Build mapper**: `fine_idx → coarse_idx`

For step 3 (stitching), the challenge is: each chunk terrain has its own `HexGrid` with indices starting at 0. To merge them, you need to:

- Determine the combined grid dimensions
- For each chunk, figure out where its hexes land in the combined grid

Since your chunks use `HexPosition` for their position, you could compute offsets in cube coordinates. Each chunk's local hex at `local_pos` maps to world position `chunk.center_hex + local_pos`.

Want to start by sketching out the chunk collection step, or jump to thinking about how to create the merged grid dimensions?

So the mapAround shows how I stitch around a single chunk to get a terrain. Does the stiching make sense for a set of chunks

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, the concept extends well! The key difference:

**mapAround**: Extract from one big terrain → one chunk-sized output

**Multi-chunk stitch**: Combine multiple chunk terrains → one merged output

Both end up with:
- A new `HexGrid` with computed dimensions
- Flattened elevation/field arrays
- An `invalidRegion` set for gaps/edges

For multi-chunk, instead of `np.pad` + slice extraction, you'd:
1. Compute bounding box from chunk positions (min/max row/col across all chunks)
2. Create empty arrays sized to fit
3. Copy each chunk's data into the correct offset location

The cube coordinates make step 3 clean: each chunk's `center_hex` tells you exactly where its local `(0,0,0)` lives in world space.

One question: In your use case, will the chunks always be contiguous (no gaps), or do we need to handle holes where a chunk wasn't generated?

So I do think we should make this work on contiguous regions it is pussible this could be in a u shape for instance in which case we would need to fill in the missing hexes. does it make sense to get the bounding box of the hexRegion and then find the chunks?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, bounding box → chunks makes a lot of sense! Benefits:

1. **Rectangular output** — much easier to work with
2. **Handles U-shapes** — includes chunks in the "gap"
3. **Cleaner math** — row/col ranges are simpler than arbitrary shapes

For the U-shape case, the hexes in the "hole" would get filled in. One question: what should happen to those fill-in hexes?

- **Option A**: Mark them in `invalidRegion` (like `mapAround` does for padding)
- **Option B**: Just include them as normal terrain (they'll have valid elevation data from the chunk)
- **Option C**: Something else?

option b. We want to save invalid for where the map can't reprsent

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call. So `invalidRegion` is only for hexes that fall outside the source terrain's actual boundaries.

Let me summarize the flow so far:

1. **HexRegion → bounding box** (row/col in coarse grid)
2. **Bounding box → set of ChunkRefs** 
3. **Load/generate each chunk** (cached)
4. **Stitch chunks → merged Terrain**
5. **Mark edge hexes as invalid** (only where coarse terrain had no data)
6. **Project watersheds** (O(n) via existing method)
7. **Build fine→coarse mapper**

Ready to start with step 1? Getting the bounding box from a `HexRegion` should be straightforward — do you want to use pixel coordinates from the hex centers, or row/col directly?

Can you build it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
from dataclasses import dataclass
from typing import Callable

@dataclass
class ZoomResult:
    """Result of zooming into a region."""
    terrain: Terrain
    basins: DrainageBasins
    mapper: Callable[[int], int]  # fine_idx → coarse_idx (-1 if invalid)
    chunks_loaded: int


def region_bounding_box(region: HexRegion) -> tuple[int, int, int, int]:
    """Get (min_row, max_row, min_col, max_col) for a region."""
    grid = region.hexGrid
    rows = []
    cols = []
    for idx in region.hexes:
        row, col = grid.index_to_row_col(idx)
        rows.append(row)
        cols.append(col)
    return min(rows), max(rows), min(cols), max(cols)


def bbox_to_chunk_refs(min_row: int, max_row: int, min_col: int, max_col: int,
                       cover: ChunkCover) -> set[ChunkRef]:
    """Find all ChunkRefs that overlap a bounding box."""
    grid = cover.terrain.hexGrid
    chunk_rings = cover.rings
    spacing = chunk_rings * 2
    
    chunks = set()
    
    # Convert row/col range to indices, then to chunk positions
    for row in range(min_row, max_row + 1, spacing):
        for col in range(min_col, max_col + 1, spacing):
            idx = grid.row_col_to_index(row, col)
            if idx < 0:
                continue
            
            # Convert to HexPosition, then to chunk position
            world_pos = grid.index_to_hexposition(idx, origin_index=0)
            chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
            chunks.add(ChunkRef(chunk_pos, chunk_rings))
    
    # Also check corners and edges to ensure coverage
    for row in [min_row, max_row]:
        for col in [min_col, max_col]:
            idx = grid.row_col_to_index(row, col)
            if idx >= 0:
                world_pos = grid.index_to_hexposition(idx, origin_index=0)
                chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
                chunks.add(ChunkRef(chunk_pos, chunk_rings))
    
    return chunks


def stitch_chunks(chunks: set[ChunkRef], cover: ChunkCover, 
                  storage: GeoStorage, scale: int) -> tuple[Terrain, dict[tuple, tuple]]:
    """Load chunks and stitch into single terrain.
    
    Returns (merged_terrain, chunk_offsets) where chunk_offsets maps
    chunk.key → (row_offset, col_offset) in merged grid.
    """
    if not chunks:
        return None, {}
    
    # Load all chunk terrains
    chunk_terrains = {}
    for chunk in chunks:
        origin_idx = cover.terrain.hexGrid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            # Chunk center is outside grid, skip
            continue
        result = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
        if result.data is not None:
            chunk_terrains[chunk.key] = result.data
    
    if not chunk_terrains:
        return None, {}
    
    # Get chunk grid dimensions (all should be same size)
    sample_terrain = next(iter(chunk_terrains.values()))
    chunk_rows = sample_terrain.hexGrid.nRows
    chunk_cols = sample_terrain.hexGrid.nCols
    chunk_radius = sample_terrain.hexGrid.radius
    
    # Find bounding box of chunks in chunk-space
    chunk_qs = [k[0] for k in chunk_terrains.keys()]
    chunk_rs = [k[1] for k in chunk_terrains.keys()]
    
    min_q, max_q = min(chunk_qs), max(chunk_qs)
    min_r, max_r = min(chunk_rs), max(chunk_rs)
    
    # Calculate merged grid size
    n_chunks_q = max_q - min_q + 1
    n_chunks_r = max_r - min_r + 1
    
    merged_rows = n_chunks_r * chunk_rows
    merged_cols = n_chunks_q * chunk_cols
    
    # Create merged terrain
    merged_grid = HexGrid(
        nRows=merged_rows, 
        nCols=merged_cols, 
        radius=chunk_radius, 
        style=sample_terrain.hexGrid.style
    )
    
    merged_terrain = Terrain(bounds=merged_grid.bounds, radius=chunk_radius)
    merged_terrain.hexGrid = merged_grid
    merged_terrain.elevations = np.full(merged_rows * merged_cols, -100.0)  # Default to water
    merged_terrain.fields = {}
    
    # Initialize fields from sample
    for field_name in sample_terrain.fields:
        merged_terrain.fields[field_name] = np.zeros(merged_rows * merged_cols)
    
    # Copy climate/geo if present
    if hasattr(sample_terrain, 'climate'):
        merged_terrain.climate = sample_terrain.climate
    if hasattr(sample_terrain, 'geo'):
        merged_terrain.geo = sample_terrain.geo
    
    # Track chunk offsets
    chunk_offsets = {}
    
    # Copy each chunk into merged terrain
    for key, chunk_terrain in chunk_terrains.items():
        q, r, s = key
        
        # Offset in merged grid
        row_offset = (r - min_r) * chunk_rows
        col_offset = (q - min_q) * chunk_cols
        chunk_offsets[key] = (row_offset, col_offset)
        
        chunk_grid = chunk_terrain.hexGrid
        
        for local_idx in range(len(chunk_terrain.elevations)):
            local_row, local_col = chunk_grid.index_to_row_col(local_idx)
            
            merged_row = row_offset + local_row
            merged_col = col_offset + local_col
            merged_idx = merged_row * merged_cols + merged_col
            
            if 0 <= merged_idx < len(merged_terrain.elevations):
                merged_terrain.elevations[merged_idx] = chunk_terrain.elevations[local_idx]
                for field_name, field_data in chunk_terrain.fields.items():
                    merged_terrain.fields[field_name][merged_idx] = field_data[local_idx]
    
    return merged_terrain, chunk_offsets


def build_fine_to_coarse_mapper(cover: ChunkCover, 
                                merged_terrain: Terrain,
                                chunk_offsets: dict,
                                scale: int) -> Callable[[int], int]:
    """Build function mapping fine terrain index → coarse terrain index."""
    coarse_grid = cover.terrain.hexGrid
    fine_grid = merged_terrain.hexGrid
    
    # Precompute for efficiency
    chunk_rows = fine_grid.nRows // len(set(k[1] for k in chunk_offsets.keys())) if chunk_offsets else fine_grid.nRows
    chunk_cols = fine_grid.nCols // len(set(k[0] for k in chunk_offsets.keys())) if chunk_offsets else fine_grid.nCols
    
    # Invert chunk_offsets for lookup
    offset_to_chunk = {v: k for k, v in chunk_offsets.items()}
    
    def mapper(fine_idx: int) -> int:
        fine_row, fine_col = fine_grid.index_to_row_col(fine_idx)
        
        # Which chunk is this in?
        chunk_row_idx = fine_row // chunk_rows
        chunk_col_idx = fine_col // chunk_cols
        
        # Find corresponding chunk key
        row_offset = chunk_row_idx * chunk_rows
        col_offset = chunk_col_idx * chunk_cols
        
        chunk_key = offset_to_chunk.get((row_offset, col_offset))
        if chunk_key is None:
            return -1
        
        q, r, s = chunk_key
        chunk_center = HexPosition(q, r, s) * (cover.rings * 2)
        
        # Local position within chunk
        local_row = fine_row - row_offset
        local_col = fine_col - col_offset
        
        # Scale down to coarse
        coarse_local_row = local_row // scale
        coarse_local_col = local_col // scale
        
        # Convert to coarse grid index
        # The chunk center in coarse grid
        center_idx = coarse_grid.hexposition_to_index(chunk_center, origin_index=0)
        if center_idx < 0:
            return -1
        
        center_row, center_col = coarse_grid.index_to_row_col(center_idx)
        
        # Offset from chunk center
        half_chunk = cover.rings
        coarse_row = center_row - half_chunk + coarse_local_row
        coarse_col = center_col - half_chunk + coarse_local_col
        
        return coarse_grid.row_col_to_index(coarse_row, coarse_col)
    
    return mapper


@patch
def zoom_region(self: ChunkCover, 
                region: HexRegion,
                scale: int = 2,
                compute_weather: bool = True) -> ZoomResult:
    """Zoom into a region with stitched chunks and projected watersheds.
    
    Args:
        region: HexRegion on the coarse terrain
        scale: Zoom factor
        compute_weather: Whether to compute weather on zoomed terrain
    
    Returns:
        ZoomResult with terrain, basins, and fine→coarse mapper
    """
    if self.db is None:
        raise ValueError("ChunkCover.db must be set for zoom_region")
    
    # Step 1: Get bounding box
    min_row, max_row, min_col, max_col = region_bounding_box(region)
    
    # Add padding for halo
    padding = self.halo_rings + 1
    min_row = max(0, min_row - padding)
    max_row = min(self.terrain.hexGrid.nRows - 1, max_row + padding)
    min_col = max(0, min_col - padding)
    max_col = min(self.terrain.hexGrid.nCols - 1, max_col + padding)
    
    # Step 2: Find chunks
    chunks = bbox_to_chunk_refs(min_row, max_row, min_col, max_col, self)
    
    # Step 3 & 4: Load and stitch
    merged_terrain, chunk_offsets = stitch_chunks(chunks, self, self.db, scale)
    
    if merged_terrain is None:
        return ZoomResult(None, None, lambda x: -1, 0)
    
    # Mark invalid regions (outside original terrain bounds)
    coarse_grid = self.terrain.hexGrid
    fine_grid = merged_terrain.hexGrid
    
    for fine_idx in range(len(merged_terrain.elevations)):
        fine_row, fine_col = fine_grid.index_to_row_col(fine_idx)
        coarse_row = fine_row // scale
        coarse_col = fine_col // scale
        
        # Check if corresponding coarse hex exists
        if coarse_row < 0 or coarse_row >= coarse_grid.nRows or \
           coarse_col < 0 or coarse_col >= coarse_grid.nCols:
            fine_grid.invalidRegion.add(fine_idx)
    
    # Step 5: Weather
    if compute_weather and hasattr(self.terrain, 'climate'):
        merged_terrain.climate = self.terrain.climate
        merged_terrain.geo = getattr(self.terrain, 'geo', None)
        merged_terrain.compute_weather()
    
    # Step 6: Lazy init coarse basin if needed
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()
    
    # Build fine_regions for watershed projection
    fine_regions = {}
    mapper = build_fine_to_coarse_mapper(self, merged_terrain, chunk_offsets, scale)
    
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue
        coarse_idx = mapper(fine_idx)
        if coarse_idx >= 0:
            if coarse_idx not in fine_regions:
                fine_regions[coarse_idx] = set()
            fine_regions[coarse_idx].add(fine_idx)
    
    # Step 7: Project watersheds
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, merged_terrain)
    
    # Build DrainageBasins
    zoomed_basins = DrainageBasins(merged_terrain, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return ZoomResult(
        terrain=merged_terrain,
        basins=zoomed_basins,
        mapper=mapper,
        chunks_loaded=len(chunk_offsets)
    )
```

This gives you:
- `zoom_region` method on `ChunkCover` 
- Uses your existing chunk cache
- Stitches multiple chunks into one terrain
- Projects watersheds O(n)
- Returns a mapper for `fine_idx → coarse_idx`

Want to test it, or should we refine any part first?

In [ ]:
#| export
from dataclasses import dataclass
from typing import Callable

@dataclass
class ZoomResult:
    """Result of zooming into a region."""
    terrain: Terrain
    basins: DrainageBasins
    mapper: Callable[[int], int]  # fine_idx → coarse_idx (-1 if invalid)
    chunks_loaded: int


def region_bounding_box(region: HexRegion) -> tuple[int, int, int, int]:
    """Get (min_row, max_row, min_col, max_col) for a region."""
    grid = region.hexGrid
    rows = []
    cols = []
    for idx in region.hexes:
        row, col = grid.index_to_row_col(idx)
        rows.append(row)
        cols.append(col)
    return min(rows), max(rows), min(cols), max(cols)


def bbox_to_chunk_refs(min_row: int, max_row: int, min_col: int, max_col: int,
                       cover: ChunkCover) -> set[ChunkRef]:
    """Find all ChunkRefs that overlap a bounding box."""
    grid = cover.terrain.hexGrid
    chunk_rings = cover.rings
    spacing = chunk_rings * 2
    
    chunks = set()
    
    # Convert row/col range to indices, then to chunk positions
    for row in range(min_row, max_row + 1, spacing):
        for col in range(min_col, max_col + 1, spacing):
            idx = grid.row_col_to_index(row, col)
            if idx < 0:
                continue
            
            # Convert to HexPosition, then to chunk position
            world_pos = grid.index_to_hexposition(idx, origin_index=0)
            chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
            chunks.add(ChunkRef(chunk_pos, chunk_rings))
    
    # Also check corners and edges to ensure coverage
    for row in [min_row, max_row]:
        for col in [min_col, max_col]:
            idx = grid.row_col_to_index(row, col)
            if idx >= 0:
                world_pos = grid.index_to_hexposition(idx, origin_index=0)
                chunk_pos, _ = world_to_chunk(world_pos, chunk_rings)
                chunks.add(ChunkRef(chunk_pos, chunk_rings))
    
    return chunks


def stitch_chunks(chunks: set[ChunkRef], cover: ChunkCover, 
                  storage: GeoStorage, scale: int) -> tuple[Terrain, dict[tuple, tuple]]:
    """Load chunks and stitch into single terrain.
    
    Returns (merged_terrain, chunk_offsets) where chunk_offsets maps
    chunk.key → (row_offset, col_offset) in merged grid.
    """
    if not chunks:
        return None, {}
    
    # Load all chunk terrains
    chunk_terrains = {}
    for chunk in chunks:
        origin_idx = cover.terrain.hexGrid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            # Chunk center is outside grid, skip
            continue
        result = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
        if result.data is not None:
            chunk_terrains[chunk.key] = result.data
    
    if not chunk_terrains:
        return None, {}
    
    # Get chunk grid dimensions (all should be same size)
    sample_terrain = next(iter(chunk_terrains.values()))
    chunk_rows = sample_terrain.hexGrid.nRows
    chunk_cols = sample_terrain.hexGrid.nCols
    chunk_radius = sample_terrain.hexGrid.radius
    
    # Find bounding box of chunks in chunk-space
    chunk_qs = [k[0] for k in chunk_terrains.keys()]
    chunk_rs = [k[1] for k in chunk_terrains.keys()]
    
    min_q, max_q = min(chunk_qs), max(chunk_qs)
    min_r, max_r = min(chunk_rs), max(chunk_rs)
    
    # Calculate merged grid size
    n_chunks_q = max_q - min_q + 1
    n_chunks_r = max_r - min_r + 1
    
    merged_rows = n_chunks_r * chunk_rows
    merged_cols = n_chunks_q * chunk_cols
    
    # Create merged terrain
    merged_grid = HexGrid(
        nRows=merged_rows, 
        nCols=merged_cols, 
        radius=chunk_radius, 
        style=sample_terrain.hexGrid.style
    )
    
    merged_terrain = Terrain(bounds=merged_grid.bounds, radius=chunk_radius)
    merged_terrain.hexGrid = merged_grid
    merged_terrain.elevations = np.full(merged_rows * merged_cols, -100.0)  # Default to water
    merged_terrain.fields = {}
    
    # Initialize fields from sample
    for field_name in sample_terrain.fields:
        merged_terrain.fields[field_name] = np.zeros(merged_rows * merged_cols)
    
    # Copy climate/geo if present
    if hasattr(sample_terrain, 'climate'):
        merged_terrain.climate = sample_terrain.climate
    if hasattr(sample_terrain, 'geo'):
        merged_terrain.geo = sample_terrain.geo
    
    # Track chunk offsets
    chunk_offsets = {}
    
    # Copy each chunk into merged terrain
    for key, chunk_terrain in chunk_terrains.items():
        q, r, s = key
        
        # Offset in merged grid
        row_offset = (r - min_r) * chunk_rows
        col_offset = (q - min_q) * chunk_cols
        chunk_offsets[key] = (row_offset, col_offset)
        
        chunk_grid = chunk_terrain.hexGrid
        
        for local_idx in range(len(chunk_terrain.elevations)):
            local_row, local_col = chunk_grid.index_to_row_col(local_idx)
            
            merged_row = row_offset + local_row
            merged_col = col_offset + local_col
            merged_idx = merged_row * merged_cols + merged_col
            
            if 0 <= merged_idx < len(merged_terrain.elevations):
                merged_terrain.elevations[merged_idx] = chunk_terrain.elevations[local_idx]
                for field_name, field_data in chunk_terrain.fields.items():
                    merged_terrain.fields[field_name][merged_idx] = field_data[local_idx]
    
    return merged_terrain, chunk_offsets


def build_fine_to_coarse_mapper(cover: ChunkCover, 
                                merged_terrain: Terrain,
                                chunk_offsets: dict,
                                scale: int) -> Callable[[int], int]:
    """Build function mapping fine terrain index → coarse terrain index."""
    coarse_grid = cover.terrain.hexGrid
    fine_grid = merged_terrain.hexGrid
    
    # Precompute for efficiency
    chunk_rows = fine_grid.nRows // len(set(k[1] for k in chunk_offsets.keys())) if chunk_offsets else fine_grid.nRows
    chunk_cols = fine_grid.nCols // len(set(k[0] for k in chunk_offsets.keys())) if chunk_offsets else fine_grid.nCols
    
    # Invert chunk_offsets for lookup
    offset_to_chunk = {v: k for k, v in chunk_offsets.items()}
    
    def mapper(fine_idx: int) -> int:
        fine_row, fine_col = fine_grid.index_to_row_col(fine_idx)
        
        # Which chunk is this in?
        chunk_row_idx = fine_row // chunk_rows
        chunk_col_idx = fine_col // chunk_cols
        
        # Find corresponding chunk key
        row_offset = chunk_row_idx * chunk_rows
        col_offset = chunk_col_idx * chunk_cols
        
        chunk_key = offset_to_chunk.get((row_offset, col_offset))
        if chunk_key is None:
            return -1
        
        q, r, s = chunk_key
        chunk_center = HexPosition(q, r, s) * (cover.rings * 2)
        
        # Local position within chunk
        local_row = fine_row - row_offset
        local_col = fine_col - col_offset
        
        # Scale down to coarse
        coarse_local_row = local_row // scale
        coarse_local_col = local_col // scale
        
        # Convert to coarse grid index
        # The chunk center in coarse grid
        center_idx = coarse_grid.hexposition_to_index(chunk_center, origin_index=0)
        if center_idx < 0:
            return -1
        
        center_row, center_col = coarse_grid.index_to_row_col(center_idx)
        
        # Offset from chunk center
        half_chunk = cover.rings
        coarse_row = center_row - half_chunk + coarse_local_row
        coarse_col = center_col - half_chunk + coarse_local_col
        
        return coarse_grid.row_col_to_index(coarse_row, coarse_col)
    
    return mapper


@patch
def zoom_region(self: ChunkCover, 
                region: HexRegion,
                scale: int = 2,
                compute_weather: bool = True) -> ZoomResult:
    """Zoom into a region with stitched chunks and projected watersheds.
    
    Args:
        region: HexRegion on the coarse terrain
        scale: Zoom factor
        compute_weather: Whether to compute weather on zoomed terrain
    
    Returns:
        ZoomResult with terrain, basins, and fine→coarse mapper
    """
    if self.db is None:
        raise ValueError("ChunkCover.db must be set for zoom_region")
    
    # Step 1: Get bounding box
    min_row, max_row, min_col, max_col = region_bounding_box(region)
    
    # Add padding for halo
    padding = self.halo_rings + 1
    min_row = max(0, min_row - padding)
    max_row = min(self.terrain.hexGrid.nRows - 1, max_row + padding)
    min_col = max(0, min_col - padding)
    max_col = min(self.terrain.hexGrid.nCols - 1, max_col + padding)
    
    # Step 2: Find chunks
    chunks = bbox_to_chunk_refs(min_row, max_row, min_col, max_col, self)
    
    # Step 3 & 4: Load and stitch
    merged_terrain, chunk_offsets = stitch_chunks(chunks, self, self.db, scale)
    
    if merged_terrain is None:
        return ZoomResult(None, None, lambda x: -1, 0)
    
    # Mark invalid regions (outside original terrain bounds)
    coarse_grid = self.terrain.hexGrid
    fine_grid = merged_terrain.hexGrid
    
    for fine_idx in range(len(merged_terrain.elevations)):
        fine_row, fine_col = fine_grid.index_to_row_col(fine_idx)
        coarse_row = fine_row // scale
        coarse_col = fine_col // scale
        
        # Check if corresponding coarse hex exists
        if coarse_row < 0 or coarse_row >= coarse_grid.nRows or \
           coarse_col < 0 or coarse_col >= coarse_grid.nCols:
            fine_grid.invalidRegion.add(fine_idx)
    
    # Step 5: Weather
    if compute_weather and hasattr(self.terrain, 'climate'):
        merged_terrain.climate = self.terrain.climate
        merged_terrain.geo = getattr(self.terrain, 'geo', None)
        merged_terrain.compute_weather()
    
    # Step 6: Lazy init coarse basin if needed
    if self.basin is None:
        self.basin = DrainageBasins(self.terrain)
        self.save()
    
    # Build fine_regions for watershed projection
    fine_regions = {}
    mapper = build_fine_to_coarse_mapper(self, merged_terrain, chunk_offsets, scale)
    
    for fine_idx in range(len(merged_terrain.elevations)):
        if fine_idx in fine_grid.invalidRegion:
            continue
        coarse_idx = mapper(fine_idx)
        if coarse_idx >= 0:
            if coarse_idx not in fine_regions:
                fine_regions[coarse_idx] = set()
            fine_regions[coarse_idx].add(fine_idx)
    
    # Step 7: Project watersheds
    zoomed_sheds = self.project_watersheds(self.basin, fine_regions, merged_terrain)
    
    # Build DrainageBasins
    zoomed_basins = DrainageBasins(merged_terrain, compute=False)
    zoomed_basins.sheds = zoomed_sheds
    
    return ZoomResult(
        terrain=merged_terrain,
        basins=zoomed_basins,
        mapper=mapper,
        chunks_loaded=len(chunk_offsets)
    )


Can you show an example where you zoom in on a watershed in the aussie_map?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Load Sydney terrain
    terrain = TerraDemo().aussie_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    # Find a medium-sized one (not too big, not too small)
    shed_sizes = [(i, len(ws.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Pick the 3rd largest
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    
    print(f"Selected watershed {target_idx}: {len(target_shed.hexes)} hexes")
    
    # Convert Watershed hexes to HexRegion
    region = HexRegion(target_shed.hexes, terrain.hexGrid)
    
    # 5. Zoom into the region
    result = cover.zoom_region(region, scale=2, compute_weather=True)
    
    print(f"\nZoom result:")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    
    # 6. Visualize
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    result.terrain.hexGrid.builder.show()
```

In [ ]:
def stitch_chunks(chunks: set[ChunkRef], cover: ChunkCover, 
                  storage: GeoStorage, scale: int) -> tuple[Terrain, dict[tuple, tuple]]:
    """Load chunks and stitch into single terrain."""
    if not chunks:
        return None, {}
    
    coarse_grid = cover.terrain.hexGrid
    
    # Load all chunk terrains and get their coarse row/col
    chunk_data = {}  # key -> (terrain, coarse_row, coarse_col)
    for chunk in chunks:
        origin_idx = coarse_grid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            continue
        result = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
        if result.data is not None:
            coarse_row, coarse_col = coarse_grid.index_to_row_col(origin_idx)
            chunk_data[chunk.key] = (result.data, coarse_row, coarse_col)
    
    if not chunk_data:
        return None, {}
    
    # Get chunk grid dimensions
    sample_terrain = next(iter(chunk_data.values()))[0]
    chunk_rows = sample_terrain.hexGrid.nRows
    chunk_cols = sample_terrain.hexGrid.nCols
    chunk_radius = sample_terrain.hexGrid.radius
    
    # Find bounding box in coarse row/col space
    coarse_rows = [d[1] for d in chunk_data.values()]
    coarse_cols = [d[2] for d in chunk_data.values()]
    
    min_coarse_row, max_coarse_row = min(coarse_rows), max(coarse_rows)
    min_coarse_col, max_coarse_col = min(coarse_cols), max(coarse_cols)
    
    # Calculate merged grid size
    # Each chunk covers (2*rings) coarse hexes, so scale up by that * scale
    coarse_span_rows = max_coarse_row - min_coarse_row + 2 * cover.rings
    coarse_span_cols = max_coarse_col - min_coarse_col + 2 * cover.rings
    
    merged_rows = coarse_span_rows * scale
    merged_cols = coarse_span_cols * scale
    
    # Create merged terrain
    merged_grid = HexGrid(
        nRows=merged_rows, 
        nCols=merged_cols, 
        radius=chunk_radius, 
        style=sample_terrain.hexGrid.style
    )
    
    merged_terrain = Terrain(bounds=merged_grid.bounds, radius=chunk_radius)
    merged_terrain.hexGrid = merged_grid
    merged_terrain.elevations = np.full(merged_rows * merged_cols, -100.0)
    merged_terrain.fields = {}
    
    for field_name in sample_terrain.fields:
        merged_terrain.fields[field_name] = np.zeros(merged_rows * merged_cols)
    
    if hasattr(sample_terrain, 'climate'):
        merged_terrain.climate = sample_terrain.climate
    if hasattr(sample_terrain, 'geo'):
        merged_terrain.geo = sample_terrain.geo
    
    chunk_offsets = {}
    
    # Copy each chunk into merged terrain
    for key, (chunk_terrain, coarse_row, coarse_col) in chunk_data.items():
        # Offset based on coarse row/col relative to min
        # Chunk center is at (coarse_row, coarse_col), but chunk extends rings in each direction
        row_offset = (coarse_row - min_coarse_row) * scale
        col_offset = (coarse_col - min_coarse_col) * scale
        chunk_offsets[key] = (row_offset, col_offset)
        
        chunk_grid = chunk_terrain.hexGrid
        
        for local_idx in range(len(chunk_terrain.elevations)):
            local_row, local_col = chunk_grid.index_to_row_col(local_idx)
            
            merged_row = row_offset + local_row
            merged_col = col_offset + local_col
            merged_idx = merged_row * merged_cols + merged_col
            
            if 0 <= merged_idx < len(merged_terrain.elevations):
                merged_terrain.elevations[merged_idx] = chunk_terrain.elevations[local_idx]
                for field_name, field_data in chunk_terrain.fields.items():
                    merged_terrain.fields[field_name][merged_idx] = field_data[local_idx]
    
    return merged_terrain, chunk_offsets


In [ ]:
@patch
def __mul__(self: HexPosition, scalar: int) -> 'HexPosition':
    return HexPosition(*(self._coords * scalar))


In [ ]:
SVGBuilder.BUILDERHIDE = True

In [ ]:
# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Load Sydney terrain
    terrain = TerraDemo().aussie_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    # Find a medium-sized one (not too big, not too small)
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    # Pick the 3rd largest
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    
    print(f"Selected watershed {target_idx}: {len(target_shed.region.hexes)} hexes")
    
    # Convert Watershed hexes to HexRegion
    region = target_shed.region
    
    # 5. Zoom into the region
    result = cover.zoom_region(region, scale=2, compute_weather=True)
    
    print(f"\nZoom result:")
    print(f"  Chunks loaded: {result.chunks_loaded}")
    print(f"  Fine terrain: {result.terrain.hexGrid.nRows}x{result.terrain.hexGrid.nCols}")
    print(f"  Watersheds: {len(result.basins.sheds)}")
    
    # 6. Visualize
    result.terrain.colorMap()
    result.terrain.hexGrid.update()
    terrain = result.terrain
    print(f"result Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
result.terrain.hexGrid.builder.show()


Much better. Can you show on the main map where this region came from

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Load Sydney terrain
    terrain = TerraDemo().aussie_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    region = target_shed.region
    
    print(f"Selected watershed {target_idx}: {len(region.hexes)} hexes")
    
    # 5. Show the coarse map with the region highlighted
    terrain.colorMap()
    
    # Highlight the selected region
    highlight_style = StyleCSS("highlight", fill="#ff000088", stroke="#ff0000", stroke_width=2)
    terrain.hexGrid.builder.add_style(highlight_style)
    
    for idx in region.hexes:
        terrain.hexGrid.hexes[idx].style = highlight_style
    
    terrain.hexGrid.update()
    
print("Coarse terrain with highlighted region:")
terrain.hexGrid.builder.show()
```

In [ ]:
SVGBuilder.BUILDERHIDE = False

In [ ]:

# Create terrain and cover with database
with GeoStorageDebugger(keep_on_error=True) as dbg:
    # 1. Load Sydney terrain
    terrain = TerraDemo().aussie_map()
    terrain.compute_weather()
    
    print(f"Terrain: {terrain.hexGrid.nRows}x{terrain.hexGrid.nCols} = {len(terrain.hexGrid.hexes)} hexes")
    
    # 2. Create cover and attach DB
    cover = ChunkCover(terrain, rings=5, halo_rings=1)
    cover.db = dbg.server
    cover.save(name="Sydney Test")
    
    # 3. Compute basins on coarse terrain
    basins = DrainageBasins(terrain)
    cover.basin = basins
    
    print(f"Found {len(basins.sheds)} watersheds")
    
    # 4. Pick a watershed and make a HexRegion
    shed_sizes = [(i, len(ws.region.hexes)) for i, ws in enumerate(basins.sheds)]
    shed_sizes.sort(key=lambda x: x[1], reverse=True)
    
    target_idx = shed_sizes[2][0] if len(shed_sizes) > 2 else 0
    target_shed = basins.sheds[target_idx]
    region = target_shed.region
    
    print(f"Selected watershed {target_idx}: {len(region.hexes)} hexes")
    
    # 5. Show the coarse map with the region highlighted
    terrain.colorMap()
    
    # Highlight the selected region
    highlight_style = StyleCSS("highlight", fill="#ff000088", stroke="#ff0000", stroke_width=2)
    terrain.hexGrid.builder.add_style(highlight_style)
    
    for idx in region.hexes:
        terrain.hexGrid.hexes[idx].style = highlight_style
    
    terrain.hexGrid.update()
    
print("Coarse terrain with highlighted region:")
terrain.hexGrid.builder.show()


r is tricky since it is there to make sure q and s sum to 0

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly! The fix is to use the chunk's `center_hex` converted to actual row/col in the coarse grid, not the cube coords directly.

Here's the corrected `stitch_chunks`:

```python
def stitch_chunks(chunks: set[ChunkRef], cover: ChunkCover, 
                  storage: GeoStorage, scale: int) -> tuple[Terrain, dict[tuple, tuple]]:
    """Load chunks and stitch into single terrain."""
    if not chunks:
        return None, {}
    
    coarse_grid = cover.terrain.hexGrid
    
    # Load all chunk terrains and get their coarse row/col
    chunk_data = {}  # key -> (terrain, coarse_row, coarse_col)
    for chunk in chunks:
        origin_idx = coarse_grid.hexposition_to_index(chunk.center_hex, origin_index=0)
        if origin_idx < 0:
            continue
        result = storage.load_or_generate_chunk(cover, origin=origin_idx, scale=scale)
        if result.data is not None:
            coarse_row, coarse_col = coarse_grid.index_to_row_col(origin_idx)
            chunk_data[chunk.key] = (result.data, coarse_row, coarse_col)
    
    if not chunk_data:
        return None, {}
    
    # Get chunk grid dimensions
    sample_terrain = next(iter(chunk_data.values()))[0]
    chunk_rows = sample_terrain.hexGrid.nRows
    chunk_cols = sample_terrain.hexGrid.nCols
    chunk_radius = sample_terrain.hexGrid.radius
    
    # Find bounding box in coarse row/col space
    coarse_rows = [d[1] for d in chunk_data.values()]
    coarse_cols = [d[2] for d in chunk_data.values()]
    
    min_coarse_row, max_coarse_row = min(coarse_rows), max(coarse_rows)
    min_coarse_col, max_coarse_col = min(coarse_cols), max(coarse_cols)
    
    # Calculate merged grid size
    # Each chunk covers (2*rings) coarse hexes, so scale up by that * scale
    coarse_span_rows = max_coarse_row - min_coarse_row + 2 * cover.rings
    coarse_span_cols = max_coarse_col - min_coarse_col + 2 * cover.rings
    
    merged_rows = coarse_span_rows * scale
    merged_cols = coarse_span_cols * scale
    
    # Create merged terrain
    merged_grid = HexGrid(
        nRows=merged_rows, 
        nCols=merged_cols, 
        radius=chunk_radius, 
        style=sample_terrain.hexGrid.style
    )
    
    merged_terrain = Terrain(bounds=merged_grid.bounds, radius=chunk_radius)
    merged_terrain.hexGrid = merged_grid
    merged_terrain.elevations = np.full(merged_rows * merged_cols, -100.0)
    merged_terrain.fields = {}
    
    for field_name in sample_terrain.fields:
        merged_terrain.fields[field_name] = np.zeros(merged_rows * merged_cols)
    
    if hasattr(sample_terrain, 'climate'):
        merged_terrain.climate = sample_terrain.climate
    if hasattr(sample_terrain, 'geo'):
        merged_terrain.geo = sample_terrain.geo
    
    chunk_offsets = {}
    
    # Copy each chunk into merged terrain
    for key, (chunk_terrain, coarse_row, coarse_col) in chunk_data.items():
        # Offset based on coarse row/col relative to min
        # Chunk center is at (coarse_row, coarse_col), but chunk extends rings in each direction
        row_offset = (coarse_row - min_coarse_row) * scale
        col_offset = (coarse_col - min_coarse_col) * scale
        chunk_offsets[key] = (row_offset, col_offset)
        
        chunk_grid = chunk_terrain.hexGrid
        
        for local_idx in range(len(chunk_terrain.elevations)):
            local_row, local_col = chunk_grid.index_to_row_col(local_idx)
            
            merged_row = row_offset + local_row
            merged_col = col_offset + local_col
            merged_idx = merged_row * merged_cols + merged_col
            
            if 0 <= merged_idx < len(merged_terrain.elevations):
                merged_terrain.elevations[merged_idx] = chunk_terrain.elevations[local_idx]
                for field_name, field_data in chunk_terrain.fields.items():
                    merged_terrain.fields[field_name][merged_idx] = field_data[local_idx]
    
    return merged_terrain, chunk_offsets
```

The key change: we now store and use `(coarse_row, coarse_col)` from `index_to_row_col(origin_idx)` instead of `(q, r)` from the cube coordinates.

should I add __mul__ to HexPosition

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, for consistency. It's a one-liner:

```python
@patch
def __mul__(self: HexPosition, scalar: int) -> 'HexPosition':
    return HexPosition(*(self._coords * scalar))
```

Same implementation as `__rmul__`, just allows `pos * 3` in addition to `3 * pos`. Most numeric-like classes define both.

## Real Data

## Code Quality Analysis

### ✅ Strengths

1. **Comprehensive caching:** Terrain, weather, watersheds all cached
2. **Smart cache keys:** Using (cover_id, origin_pos, scale) uniquely identifies chunks
3. **Automatic invalidation:** Ensures consistency when coarse data changes
4. **Position-based queries:** Using HexPosition prevents grid alignment issues
5. **Efficient watershed projection:** O(n×k²) not O(n²)
6. **Result objects:** Consistent LoadResult/SaveResult pattern
7. **Testing infrastructure:** GeoStorageDebugger for test isolation

### ⚠️ Concerns

1. **Cache Size Management**
   - No LRU eviction
   - No max cache size limit
   - Could grow unbounded in long-running apps

2. **Chunk Reconstruction**
   - `_reconstruct_terrain_from_rows()` infers grid dimensions from data
   - Could fail if chunk has irregular shape
   - No validation of reconstructed grid

3. **Transaction Safety**
   - Some operations use `with self.db.conn:` (good)
   - Others don't (potential partial writes)
   - Invalidation is multi-statement (not atomic)

4. **Error Handling**
   - Generic `except Exception as e:` catches everything
   - Could hide bugs (e.g., programming errors vs. data errors)
   - No retry logic for transient failures

5. **Missing Features**
   - No access time tracking (needed for LRU)
   - No cache statistics (hit rate, size, age)
   - No prefetching hints
   - No background cache warming

---

## Suggestions for Improvement

### 1. Cache Management

#### Add LRU Eviction
```python
@dataclass
class CacheStats:
    cover_id: int
    chunk_count: int
    total_hexes: int
    oldest_access: int
    newest_access: int
    total_size_mb: float

@patch
def get_cache_stats(self: GeoStorage, cover_id: int = None) -> CacheStats:
    """Get statistics about cached chunks."""
    # Count chunks, hexes, compute size
    # Track access times (need to add last_accessed column)

@patch
def evict_lru_chunks(self: GeoStorage, cover_id: int, 
                     target_size_mb: float = 100) -> SaveResult:
    """Evict least-recently-accessed chunks until under size limit."""
    # Requires: ALTER TABLE hex_data ADD COLUMN last_accessed INTEGER
    # Query chunks by access time
    # Delete oldest until size < target
```

#### Add Access Tracking
```python
# Schema change needed:
@dataclass
class HexData:
    # ... existing ...
    last_accessed: int = 0  # NEW: Unix timestamp
    access_count: int = 0   # NEW: For popularity tracking

# Update on load:
@patch
def _load_cached_chunk(self: GeoStorage, ...):
    # ... load data ...
    
    # Update access stats
    now = int(datetime.now().timestamp())
    self.db.execute("""
        UPDATE hex_data 
        SET last_accessed = ?, access_count = access_count + 1
        WHERE world_id = ? AND chunk_q = ? AND chunk_r = ? AND chunk_s = ?
          AND scale_level = ?
    """, [now, cover_id, origin_pos.q, origin_pos.r, origin_pos.s, scale])
```

### 2. Chunk Reconstruction Safety

#### Validate Reconstructed Grid
```python
@patch
def _reconstruct_terrain_from_rows(self: GeoStorage, rows: list[dict], 
                                   cover_id: int) -> Terrain:
    # ... existing logic ...
    
    # VALIDATE reconstructed grid
    expected_hex_count = len(rows)
    actual_hex_count = len(grid.hexes)
    
    if actual_hex_count != expected_hex_count:
        raise ValueError(
            f"Grid reconstruction mismatch: {actual_hex_count} grid hexes "
            f"vs {expected_hex_count} data rows. Chunk may be incomplete."
        )
    
    # Verify no gaps in coverage
    positions_in_rows = {(r['q'], r['r'], r['s']) for r in rows}
    positions_in_grid = {
        (grid.index_to_hexposition(i).q, 
         grid.index_to_hexposition(i).r,
         grid.index_to_hexposition(i).s)
        for i in range(len(grid.hexes))
    }
    
    missing = positions_in_grid - positions_in_rows
    if missing:
        print(f"⚠ Warning: {len(missing)} hexes missing data (will be zero)")
```

#### Store Grid Metadata
```python
# Alternative: Store grid params explicitly

@dataclass
class CachedChunk:
    """Metadata for cached chunks."""
    id: int = None
    cover_id: int = 0
    chunk_q, chunk_r, chunk_s: int
    scale: int = 2
    # Store grid reconstruction params
    nrows: int = 0
    ncols: int = 0
    radius: float = 0.0
    hex_count: int = 0
    created, last_accessed, access_count: int

# Then use stored params for reconstruction
```

### 3. Transaction Safety

#### Wrap Multi-Statement Operations
```python
@patch
def invalidate_all_caches(self: GeoStorage, cover_id: int) -> dict:
    """Invalidate all cached data when coarse terrain changes."""
    results = {}
    
    # Use transaction for atomic invalidation
    try:
        with self.db.conn:
            # All deletes in one transaction
            cursor = self.db.execute(
                "DELETE FROM hex_data WHERE world_id = ? AND scale_level > 0",
                [cover_id]
            )
            results['terrain'] = SaveResult(cursor.rowcount, 'invalidated', ...)
            
            cursor = self.db.execute(
                "DELETE FROM hex_weather WHERE world_id = ? AND scale_level > 0",
                [cover_id]
            )
            results['weather'] = SaveResult(cursor.rowcount, 'invalidated', ...)
            
            # ... more deletes ...
            
        return results
    
    except Exception as e:
        # Transaction rolled back automatically
        return {'error': SaveResult(None, 'error', str(e))}
```

### 4. Error Handling Refinement

#### Distinguish Error Types
```python
class CacheError(Exception):
    """Recoverable cache errors."""

class DataIntegrityError(Exception):
    """Unrecoverable data corruption."""

@patch
def load_or_generate_chunk(self: GeoStorage, ...):
    try:
        # Check cache
        cached = self._load_cached_chunk(...)
        if cached.status == 'loaded':
            return LoadResult(cached.data, 'cached', cached.context)
    
    except DataIntegrityError as e:
        # Log and invalidate bad cache entry
        print(f"⚠ Cache corruption detected: {e}")
        self.invalidate_chunk_cache(cover.ident)
        # Fall through to regeneration
    
    except CacheError as e:
        # Transient error, could retry
        print(f"⚠ Cache lookup failed: {e}")
        # Fall through to regeneration
    
    # Generate fresh
    try:
        zoomed = cover.zoomChunkCombined(...)
        self._save_chunk_cache(...)
        return LoadResult(zoomed, 'generated', ...)
    
    except Exception as e:
        # Generation failed - this is serious
        return LoadResult(None, 'error', f'Generation failed: {e}')
```

### 5. Background Cache Operations

#### Prefetch Adjacent Chunks
```python
@patch
def prefetch_neighbors(self: GeoStorage, cover: ChunkCover, 
                       origin: int, scale: int) -> list[SaveResult]:
    """Background prefetch of neighboring chunks."""
    grid = cover.terrain.hexGrid
    origin_pos = grid.index_to_hexposition(origin)
    
    neighbors = []
    for direction in HexPosition.directions():
        neighbor_origin = origin_pos + (direction * cover.distance)
        neighbor_idx = grid.hexposition_to_index(neighbor_origin, origin_index=0)
        
        if neighbor_idx >= 0:
            neighbors.append((neighbor_idx, neighbor_origin))
    
    # Generate in background (could use ThreadPoolExecutor)
    results = []
    for idx, pos in neighbors:
        if not self.has_cached_chunk(cover.ident, pos, scale):
            result = self.load_or_generate_chunk(cover, idx, scale)
            results.append(result)
    
    return results
```

### 6. Cache Statistics Dashboard

#### Implement Monitoring
```python
@dataclass
class CacheMetrics:
    cover_id: int
    total_chunks: int
    total_hexes: int
    terrain_size_mb: float
    weather_size_mb: float
    watershed_size_mb: float
    hit_rate: float         # hits / (hits + misses)
    avg_age_hours: float
    oldest_chunk_age_hours: float

@patch
def get_cache_metrics(self: GeoStorage, cover_id: int) -> CacheMetrics:
    """Comprehensive cache statistics."""
    # Query chunk counts, sizes, access patterns
    # Compute hit rate from access_count and generation timestamps
    # Calculate ages from created/last_accessed

@patch
def print_cache_report(self: GeoStorage, cover_id: int):
    """Human-readable cache report."""
    metrics = self.get_cache_metrics(cover_id)
    
    print(f"Cache Report for Cover {cover_id}")
    print(f"  Chunks: {metrics.total_chunks} ({metrics.total_hexes} hexes)")
    print(f"  Size: {metrics.terrain_size_mb + metrics.weather_size_mb:.1f} MB")
    print(f"    - Terrain: {metrics.terrain_size_mb:.1f} MB")
    print(f"    - Weather: {metrics.weather_size_mb:.1f} MB")
    print(f"    - Watersheds: {metrics.watershed_size_mb:.1f} MB")
    print(f"  Hit Rate: {metrics.hit_rate:.1%}")
    print(f"  Age: avg {metrics.avg_age_hours:.1f}h, max {metrics.oldest_chunk_age_hours:.1f}h")
```

---

## Documentation Updates Needed

### 1. Update DatabaseSchema.md

Add sections:
- ✅ **ChunkCover Integration** - save/load, persistence
- ✅ **Chunk Caching** - cache keys, hit rates, invalidation
- ✅ **Weather Caching** - seasonal caching per chunk
- ✅ **Watershed Persistence** - assignment caching, projection
- ✅ **New Tables** - WatershedMeta, ChunkBorder
- ✅ **Schema Extensions** - updated HexData, HexWeather fields

### 2. Update DATABASE_QUICK_REF.md

Add examples:
```python
# ChunkCover workflow
cover = ChunkCover(master, rings=5)
cover.db = storage
cover.save(name="Master")

# Cached zooming
zoomed = cover.zoom_cached(origin=middle, scale=4)  # Generates + caches
zoomed2 = cover.zoom_cached(origin=middle, scale=4) # Cache hit!

# Full data zoom (terrain + weather + watersheds)
full = cover.zoom_with_full_data(origin=middle, scale=4)

# Cache management
storage.invalidate_all_caches(cover.ident)
stats = storage.get_cache_stats(cover.ident)
```

How much bigger can these chunked worlds be given our 3-6k size limit on view ports


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question! Let's do the math:

**Current Test Numbers:**
- Maui master: 100×70 = **7,000 coarse hexes**
- Zoomed chunk (scale=2): 42×42 = **1,764 fine hexes** per viewport

**Viewport Capacity (3-6k hexes):**
- At scale=2: can show ~2-3 adjacent chunks
- At scale=4: chunk would be ~84×84 = **7,056 fine hexes** (right at your limit)

**World Size Calculation:**

| Coarse Grid | Scale | Effective Fine Grid | Ratio to Viewport |
|-------------|-------|---------------------|-------------------|
| 100×100 (10k) | 4× | 400×400 (160k) | **~30× larger** |
| 500×500 (250k) | 4× | 2000×2000 (4M) | **~800× larger** |
| 1000×1000 (1M) | 4× | 4000×4000 (16M) | **~3000× larger** |

**Practical Limits:**

1. **Coarse terrain generation**: O(n²) for watersheds - gets slow above ~500×500
2. **Memory for coarse map**: ~250k hexes fits easily in RAM
3. **DB cache size**: Your eviction caps at 100 chunks = ~180k cached fine hexes

**Recommendation:**
A **500×500 coarse map** with **scale=4 zooming** gives you a **~4 million fine-hex world** while keeping:
- Coarse generation reasonable (~minutes)
- Viewport at 3-6k hexes
- ~30-50× zoom-out overview available

That's roughly the size of **all of Japan at ~250m/hex resolution** for the fine view!